<a href="https://colab.research.google.com/github/nikamsudarshan/E-commerce-Product-Recommender/blob/main/notebooks/market_basket_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

print("Downloading Transaction Dataset...")
# We use a reliable, public groceries dataset hosted on GitHub for instant Colab loading
url = "https://raw.githubusercontent.com/amankharwal/Website-data/master/Groceries_dataset.csv"
df = pd.read_csv(url)

print("Dataset loaded! Shape:", df.shape)
print("\n--- Raw Data Sample ---")
print(df.head(3))

# --- Preprocessing: Grouping by Transaction ---
print("\nProcessing Data into Market Baskets...")
# A single transaction is uniquely identified by the Member ID and the Date of purchase
df['single_transaction'] = df['Member_number'].astype(str) + '_' + df['Date'].astype(str)

# Create a cross-tabulation where rows are unique baskets and columns are individual items
basket = (df.groupby(['single_transaction', 'itemDescription'])['itemDescription']
          .count().unstack().reset_index().fillna(0)
          .set_index('single_transaction'))

# The Apriori algorithm requires strictly boolean data (True/False).
# If a user bought 5 apples, we only care that apples are in the basket (True), not the quantity.
def encode_units(x):
    if x <= 0:
        return False
    if x >= 1:
        return True

# Apply the boolean encoding to the entire grid
basket_sets = basket.map(encode_units)

print(f"\nSuccessfully created {basket_sets.shape[0]} unique baskets containing {basket_sets.shape[1]} different items.")

Dataset loaded! Shape: (38765, 3)

--- Raw Data Sample ---
   Member_number        Date itemDescription
0           1808  21-07-2015  tropical fruit
1           2552  05-01-2015      whole milk
2           2300  19-09-2015       pip fruit

Processing Data into Market Baskets...

Successfully created 14963 unique baskets containing 167 different items.


In [2]:
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

print("Running Apriori Algorithm...")
# 1. Find frequent itemsets (min_support = 0.002 means ~30 baskets out of 14,963)
frequent_itemsets = apriori(basket_sets, min_support=0.002, use_colnames=True)
print(f"Found {len(frequent_itemsets)} mathematically significant item combinations.")

print("Generating Association Rules...")
# 2. Generate the rules, filtering out anything with a Lift less than 1.0
# (Lift < 1 means the items are actually bought together LESS often than random chance)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

# 3. Clean up the output to make it human-readable (converting frozensets to strings)
rules['antecedents'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequents'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

# 4. Isolate the most powerful rules by sorting by Lift and Confidence
strong_rules = rules.sort_values(by=['lift', 'confidence'], ascending=[False, False])

print("\n--- Top 10 Hidden Purchasing Patterns ---")
# Displaying the final metrics rounded to 3 decimal places
final_output = strong_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10).round(3)
print(final_output.to_string(index=False))

Running Apriori Algorithm...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Found 330 mathematically significant item combinations.
Generating Association Rules...

--- Top 10 Hidden Purchasing Patterns ---
      antecedents       consequents  support  confidence  lift
             curd           sausage    0.003       0.087 1.447
          sausage              curd    0.003       0.049 1.447
      brown bread       canned beer    0.002       0.064 1.363
      canned beer       brown bread    0.002       0.051 1.363
frozen vegetables           sausage    0.002       0.074 1.226
          sausage frozen vegetables    0.002       0.034 1.226
     bottled beer           sausage    0.003       0.074 1.222
          sausage      bottled beer    0.003       0.055 1.222
      frankfurter  other vegetables    0.005       0.136 1.116
 other vegetables       frankfurter    0.005       0.042 1.116


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag